In [ ]:
import sys, warnings, time
sys.path.append("..")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import yfinance as yf
import tensorflow as tf
from tensorflow import keras  # type: ignore
from arch import arch_model
from arch.bootstrap import MCS

TICKERS = ["^GSPC"]                       # add e.g. "AAPL", "JPM" if you like (about 5 min per asset)
START = "2010-01-01"
N_SEEDS = 5
REFIT_EVERY = 20                          # HAR re-estimation frequency (days) within each fold
LOOK = 22
FOLD_STARTS = [0.5, 0.6, 0.7, 0.8, 0.9]   # training ends at these fractions of the data
FOLD_LEN = 0.1                            # each test block = next 10% of the data

In [ ]:
_cache = {}
def load_data(ticker):
    """Download OHLC once; returns (log returns in %, OHLC frame aligned to the returns)."""
    if ticker not in _cache:
        px = yf.download(ticker, start=START, auto_adjust=True, progress=False)
        if isinstance(px.columns, pd.MultiIndex):
            px.columns = px.columns.get_level_values(0)
        px = px[["Open", "High", "Low", "Close"]].dropna()
        r = (100 * np.log(px["Close"] / px["Close"].shift(1))).dropna()
        r.name = "log_return"
        _cache[ticker] = (r, px)
    return _cache[ticker]

def load_returns(ticker):
    return load_data(ticker)[0]

def range_proxy(r, px):
    """Garman-Klass + squared overnight return, rescaled by a trailing 252-day scale factor
    (shifted one day, backfilled) -- same construction as cell 2 of notebook 11."""
    lnHL = np.log(px["High"] / px["Low"])
    lnCO = np.log(px["Close"] / px["Open"])
    lnON = np.log(px["Open"] / px["Close"].shift(1))
    gk = 1e4 * (0.5 * lnHL ** 2 - (2 * np.log(2) - 1) * lnCO ** 2)
    gk_on = (gk + 1e4 * lnON ** 2).loc[r.index]
    sq_ret = r ** 2
    roll_scale = (sq_ret.rolling(252, min_periods=126).sum()
                  / gk_on.rolling(252, min_periods=126).sum()).shift(1).bfill()
    return (roll_scale * gk_on).clip(lower=1e-6)

SPECS = {
    "GARCH(1,1)": dict(vol="GARCH", p=1, q=1),
    "EGARCH":     dict(vol="EGARCH", p=1, o=1, q=1),
    "GJR-GARCH":  dict(vol="GARCH", p=1, o=1, q=1),
}

def garch_forecasts(r, split, end):
    """Estimate on r[:split] only; one-day-ahead forecasts for the test block r[split:end]."""
    gf_all, test = {}, {}
    for name, spec in SPECS.items():
        res = arch_model(r, mean="Constant", dist="t", **spec).fit(last_obs=r.index[split - 1], disp="off")
        gf = res.forecast(horizon=1, start=r.index[0], reindex=False).variance["h.1"]  # made at t, for t+1
        gf_all[name] = gf
        test[name] = gf.shift(1).loc[r.index[split:end]]
    test["Rolling 21d var"] = r.rolling(21).var().shift(1).loc[r.index[split:end]]
    return gf_all, pd.DataFrame(test)

In [3]:
def qlike_loss(y_true, y_pred):
    y_, base = y_true[:, 0:1], y_true[:, 1:2]
    out = tf.clip_by_value(y_pred, -4.0, 4.0)
    f = base * tf.exp(out)
    return tf.reduce_mean(tf.math.log(f) + y_ / f)

def build_model(n_feat, lr=3e-4):
    m = keras.Sequential([
        keras.layers.Input(shape=(LOOK, n_feat)),
        keras.layers.LSTM(16),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(1, kernel_initializer="zeros", bias_initializer="zeros"),
    ])
    m.compile(optimizer=keras.optimizers.Adam(lr), loss=qlike_loss)
    return m

def train_ensemble(s, n_seeds):
    preds = []
    for seed in range(n_seeds):
        np.random.seed(seed); tf.random.set_seed(seed)
        m = build_model(s["X_tr"].shape[2])
        early = keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True)
        m.fit(s["X_tr"], s["Y_tr"], validation_data=(s["X_val"], s["Y_val"]),
              epochs=200, batch_size=64, callbacks=[early], verbose=0)
        out = np.clip(m.predict(s["X_te"], verbose=0).ravel(), -4, 4)
        preds.append(s["Y_te"][:, 1] * np.exp(out))
    return np.array(preds).mean(axis=0)

def nn_forecasts(r, gjr_gf, split, end, n_seeds):
    feat = pd.DataFrame({
        "ret": r, "sq_ret": r ** 2,
        "abs_mean_5d": r.abs().rolling(5).mean(),
        "rv_21d": r.rolling(21).std() * np.sqrt(252),
        "log_garch_var": np.log(gjr_gf),
    }).dropna()
    split_date = r.index[split]
    end_date = r.index[end] if end < len(r) else r.index[-1] + pd.Timedelta(days=1)

    tr = feat.index < split_date                       # scaling uses training data only
    mu, sd = feat[tr].mean(), feat[tr].std()
    feat_s = ((feat - mu) / sd).clip(-5, 5)
    vals, idx = feat_s.values, feat_s.index
    rsq = (r ** 2).loc[idx].values
    gv = gjr_gf.loc[idx].values

    X, y, g, dates = [], [], [], []
    for t in range(LOOK, len(idx)):
        X.append(vals[t - LOOK:t]); y.append(rsq[t]); g.append(gv[t - 1]); dates.append(idx[t])
    X, y, g, dates = np.array(X), np.array(y), np.array(g), pd.DatetimeIndex(dates)

    train_mask = dates < split_date
    test_mask = (dates >= split_date) & (dates < end_date)
    const_var = (r ** 2)[r.index < split_date].mean()

    def make_sets(X_, base):
        Y = np.column_stack([y, base])
        Xa, Ya = X_[train_mask], Y[train_mask]
        v = int(len(Xa) * 0.9)
        return dict(X_tr=Xa[:v], Y_tr=Ya[:v], X_val=Xa[v:], Y_val=Ya[v:],
                    X_te=X_[test_mask], Y_te=Y[test_mask])

    out = pd.DataFrame(index=dates[test_mask])
    out["Hybrid"] = train_ensemble(make_sets(X, g), n_seeds)
    out["LSTM"] = train_ensemble(make_sets(X[:, :, :4], np.full(len(y), const_var)), n_seeds)
    return out

In [ ]:
def har_design(x):
    """Row t holds information up to and including day t; it forecasts day t+1."""
    return pd.DataFrame({"daily": x, "weekly": x.rolling(5).mean(), "monthly": x.rolling(22).mean()})

def rolling_har(series, split, end, log=False):
    """HAR on a variance proxy, re-estimated every REFIT_EVERY days inside the fold [split, end).
    Each fit uses only origins whose target (t+1) is before the first day it forecasts."""
    x = np.log(series) if log else series
    X = sm.add_constant(har_design(x))
    y_next = x.shift(-1)
    preds = []
    for s in range(split, end, REFIT_EVERY):
        e = min(s + REFIT_EVERY, end)
        Xt, yt = X.iloc[: s - 1], y_next.iloc[: s - 1]
        ok = Xt.notna().all(axis=1) & yt.notna()
        res = sm.OLS(yt[ok], Xt[ok]).fit()
        p = res.predict(X.iloc[s - 1: e - 1])
        if log:
            p = np.exp(p + 0.5 * res.mse_resid)
        preds.append(pd.Series(p.values, index=series.index[s:e]))
    return pd.concat(preds)

def run_fold(r, proxy_rb, split_frac):
    n = len(r)
    split = int(n * split_frac)
    end = min(int(n * (split_frac + FOLD_LEN)), n)
    gf_all, fc = garch_forecasts(r, split, end)
    nn = nn_forecasts(r, gf_all["GJR-GARCH"], split, end, N_SEEDS)
    floor = 0.01 * proxy_rb.iloc[:split].mean()
    har = rolling_har(proxy_rb, split, end).clip(lower=floor).rename("HAR")
    loghar = rolling_har(proxy_rb, split, end, log=True).rename("log-HAR")
    out = fc.join(nn, how="inner").join(har, how="inner").join(loghar, how="inner")
    return out.dropna().clip(lower=1e-8)

folds, proxies = {}, {}
for tk in TICKERS:
    r, px = load_data(tk)
    proxy_rb = range_proxy(r, px)
    proxies[tk] = {"Squared returns": r ** 2, "Range-based": proxy_rb}
    print(f"{tk}: mean r^2 / mean range proxy = {(r ** 2).mean() / proxy_rb.mean():.3f}")
    folds[tk] = []
    for f in FOLD_STARTS:
        t0 = time.time()
        fc = run_fold(r, proxy_rb, f)
        folds[tk].append(fc)
        print(f"{tk} fold {f:.1f}: test {fc.index[0].date()} to {fc.index[-1].date()} "
              f"({len(fc)} days), {time.time() - t0:.0f}s")


In [ ]:
def fold_table(tk, proxy_name):
    rows = {}
    for fc in folds[tk]:
        proxy = proxies[tk][proxy_name].loc[fc.index]
        rows[f"{fc.index[0].date()} to {fc.index[-1].date()}"] = fc.apply(lambda f: (np.log(f) + proxy / f).mean())
    return pd.DataFrame(rows).T

tables = {(tk, pn): fold_table(tk, pn) for tk in TICKERS for pn in proxies[tk]}
for (tk, pn), tab in tables.items():
    print(f"{tk} - QLIKE by test block, {pn} proxy (lower is better)")
    display(tab.round(4))
    rel = tab.sub(tab["GJR-GARCH"], axis=0)
    print("Folds in which each model beats GJR-GARCH: " +
          ", ".join(f"{m} {(rel[m] < 0).sum()}/{len(rel)}" for m in rel.columns if m != "GJR-GARCH") + "\n")


In [ ]:
def dm_test(L_a, L_b, lags=5):
    d = (L_a - L_b).values
    res = sm.OLS(d, np.ones(len(d))).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return res.tvalues[0], res.pvalues[0]

def summarize(losses, label):
    print(f"--- {label} ---")
    print("QLIKE:")
    display(losses.mean().sort_values().round(4).to_frame("QLIKE"))
    print("Diebold-Mariano vs GJR-GARCH (negative = model has lower loss):")
    for m in losses.columns:
        if m == "GJR-GARCH":
            continue
        s_, p_ = dm_test(losses[m], losses["GJR-GARCH"])
        print(f"  {m:16s} DM = {s_:6.2f}   p = {p_:.3f}")
    mcs = MCS(losses, size=0.10, reps=1000, seed=42)
    mcs.compute()
    print("90% Model Confidence Set:", list(mcs.included), "\n")

pooled_all = {}
for tk in TICKERS:
    pooled = pd.concat(folds[tk])
    pooled_all[tk] = pooled
    print(f"===== {tk}: pooled over {len(pooled)} test days =====")
    for pn, p in proxies[tk].items():
        proxy = p.loc[pooled.index]
        summarize(pooled.apply(lambda f: np.log(f) + proxy / f), f"{pn} proxy")


In [ ]:
# COVID fold: the test block that contains March 2020
for tk in TICKERS:
    k = [i for i, fc in enumerate(folds[tk]) if fc.index[0] <= pd.Timestamp("2020-03-16") <= fc.index[-1]]
    assert len(k) == 1, "no unique fold contains 2020-03-16"
    fc = folds[tk][k[0]]
    print(f"===== {tk}: COVID fold, {fc.index[0].date()} to {fc.index[-1].date()} ({len(fc)} days) =====")
    for pn, p in proxies[tk].items():
        proxy = p.loc[fc.index]
        summarize(fc.apply(lambda f: np.log(f) + proxy / f), f"{pn} proxy")


In [ ]:
for (tk, pn), tab in tables.items():
    rel = tab[["GARCH(1,1)", "EGARCH", "LSTM", "Hybrid", "HAR", "log-HAR"]].sub(tab["GJR-GARCH"], axis=0)
    ax = rel.plot(kind="bar", figsize=(12, 5), width=0.8)
    ax.axhline(0, color="black", lw=1)
    ax.set_ylabel("QLIKE minus GJR-GARCH QLIKE (lower = better)")
    ax.set_title(f"{tk}: QLIKE relative to GJR-GARCH, by test block ({pn} proxy)")
    plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()